In [1]:
from IPython.display import display, Markdown

from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult, DocumentContentFormat

from concurrent.futures import ThreadPoolExecutor
from pdf2image import convert_from_bytes


from modules.samples.app_settings import AppSettings
from modules.samples.utils.stopwatch import Stopwatch
from modules.samples.utils.storage_utils import create_json_file
from modules.samples.models.document_processing_result import DataExtractionResult


from modules.samples.models.vehicle_insurance_policy import VehicleInsurancePolicy
from modules.samples.models.bir_2307 import Bir2307
from modules.samples.confidence.confidence_utils import merge_confidence_values
from modules.samples.confidence.openai_confidence import evaluate_confidence as evaluate_openai_confidence
from modules.samples.confidence.document_intelligence_confidence import evaluate_confidence as evaluate_di_confidence
from modules.samples.evaluation.accuracy_evaluator import AccuracyEvaluator
from modules.samples.evaluation.comparison import get_extraction_comparison

In [2]:
from dotenv import dotenv_values


settings = AppSettings(dotenv_values(f".env"))

In [3]:
openai_client = AzureOpenAI(
    api_key=settings.azure_openai_api_key,
    api_version="2025-02-01-preview",
    azure_endpoint=settings.azure_openai_endpoint,
    azure_deployment=settings.gpt4o_model_deployment_name,
)

document_intelligence_client = DocumentIntelligenceClient(
    endpoint=settings.document_intelligence_endpoint,
    credential=AzureKeyCredential(settings.document_intelligence_api_key),
)

### Establish the expected output

To compare the accuracy of the extraction process, the expected output of the extraction process has been defined in the following code block based on each page of a [Vehicle Insurance Policy](../../../assets/vehicle_insurance/policy_1.pdf).

> **Note**: More insurance policy examples can be found in the [assets folder](../../../assets/vehicle_insurance). These examples include the PDF file and an associated JSON metadata file that provides the expected structured output. You can add your own scenarios by following the same structure.

The expected output has been defined by a human evaluating the document.

In [22]:
import os
import json

working_dir = os.path.abspath('./')

path = f"{working_dir}/files/"
metadata_fname = "BIR2307_AHC_TMI_0012327_0823_20240205.json" # Change this to the file you want to evaluate
metadata_fpath = f"{path}{metadata_fname}"

with open(metadata_fpath, 'r') as f:
    data = json.load(f)


from modules.samples.models.bir_2307 import Bir2307
    

pdf_fname = data['fname']
pdf_fpath = f"{path}{pdf_fname}"

bir_2307_evaluator = AccuracyEvaluator(match_keys=[])

In [5]:
expected = Bir2307(**data['expected'])

## Extract data from the document

The following code block executes the data extraction process using Azure OpenAI's GPT-4o model using vision capabilities.

It performs the following steps:

1. Get the document bytes from the provided file path. _Note: In this example, we are processing a local document, however, you can use any document storage location of your choice, such as Azure Blob Storage._
2. Use Azure AI Document Intelligence to analyze the structure of the document and convert it to Markdown format using the pre-built layout model.
3. Use pdf2image to convert the document's pages into images per page as base64 strings.
4. Using Azure OpenAI's GPT-4o model and its [Structured Outputs feature](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/structured-outputs?tabs=python-secure), extract a structured data transfer object (DTO) from the content of the images.

In [7]:
with Stopwatch() as di_stopwatch:
    with open(pdf_fpath, "rb") as f:
        poller = document_intelligence_client.begin_analyze_document(
            model_id="prebuilt-layout",
            body=f,
            output_content_format=DocumentContentFormat.MARKDOWN,
            content_type="application/pdf"
        )
    
    result: AnalyzeResult = poller.result()

markdown = result.content

In [8]:
markdown

'<!-- PageHeader="TS-WF-205F-0012327" -->\n\n\n<figure>\n\nRepublic of the Philippines\nDepartment of Finance\nBureau of Internal Revenue\n\n</figure>\n\n\nFor BIR BCS/\nUse Only Item:\n\nBIR Form No.\n2307\nJanuary 2018 (ENCS)\n\n\n# Certificate of Creditable Tax Withheld At Source\n\n2307 01/18ENCS\n\nFill in all applicable spaces. Mark all appropriate boxes with an "X".\n\n1 For the Period\nFrom\n07012023\n\n(MM/DD/YY)\n\nTo\n09302023\n\n(MM/DD/YY)\n\n\n## Part I - Payee Information\n\n2 Taxpayer Identification Number (TIN)\n\n267-090-070-00000\n\n3 Payee\'s Name (Last Name, First Name, Middle Name for Individual OR Registered Name for Non-Individuals)\n\nTHERMA MARINE, INC.\n\n4 Registered Address\nMOBILE 2, LAWIS, SANTA ANA, AGUSAN DEL NORTE PHILIPPINES 8602 PHILIPPINES\n\n4A ZIP Code\n\n8602\n\n5 Foreign Address, if applicable\n\n\n## Part II - Payor Information\n\n6 Taxpayer Identification Number (TIN)\n\n008-657-558-0000\n\n7 Payor\'s Name (Last Name, First Name, Middle Name fo

In [9]:
markdown = '<!-- PageHeader="TS-WF-205F-0012327" -->\n\n\n<figure>\n\nRepublic of the Philippines\nDepartment of Finance\nBureau of Internal Revenue\n\n</figure>\n\n\nFor BIR BCS/\nUse Only Item:\n\nBIR Form No.\n2307\nJanuary 2018 (ENCS)\n\n\n# Certificate of Creditable Tax Withheld At Source\n\n2307 01/18ENCS\n\nFill in all applicable spaces. Mark all appropriate boxes with an "X".\n\n1 For the Period\nFrom\n07012023\n\n(MM/DD/YY)\n\nTo\n09302023\n\n(MM/DD/YY)\n\n\n## Part I - Payee Information\n\n2 Taxpayer Identification Number (TIN)\n\n267-090-070-00000\n\n3 Payee\'s Name (Last Name, First Name, Middle Name for Individual OR Registered Name for Non-Individuals)\n\nTHERMA MARINE, INC.\n\n4 Registered Address\nMOBILE 2, LAWIS, SANTA ANA, AGUSAN DEL NORTE PHILIPPINES 8602 PHILIPPINES\n\n4A ZIP Code\n\n8602\n\n5 Foreign Address, if applicable\n\n\n## Part II - Payor Information\n\n6 Taxpayer Identification Number (TIN)\n\n008-657-558-0000\n\n7 Payor\'s Name (Last Name, First Name, Middle Name for Individual OR Registered Name for Non-Individuals)\nANGAT HYDROPOWER CORPORATION\n\n8 Registered Address\nANGAT HYDROELECTRIC POWER PLANT SAN LORENZO, NORZAGARAY, BULACAN\n\n8A ZIP Code\n\n3013\n\n\n<table>\n<tr>\n<th colspan="7">Part III - Details of Monthly Income Payments and Taxes Withheld</th>\n</tr>\n<tr>\n<th rowspan="2">Income Payments Subject to Expanded Withholding Tax</th>\n<th rowspan="2">ATC</th>\n<th colspan="4">AMOUNT OF INCOME PAYMENTS</th>\n<th rowspan="2">Tax Withheld for the Quarter</th>\n</tr>\n<tr>\n<th>1st Month of the Quarter</th>\n<th>2nd Month of the Quarter</th>\n<th>3rd Month of the Quarter</th>\n<th>Total</th>\n</tr>\n<tr>\n<td>EWT- Income payments made by top 10,000 private corporations to their local/resident supplier of services</td>\n<td>WC 160</td>\n<td></td>\n<td>1.22</td>\n<td></td>\n<td>1.22</td>\n<td>0.02</td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td>Total</td>\n<td></td>\n<td></td>\n<td>1.22</td>\n<td></td>\n<td>1.22</td>\n<td>0.02</td>\n</tr>\n<tr>\n<td>Money Payments Subject to Withholding of Business Tax (Government &amp; Private)</td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n<tr>\n<td>Total</td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>\n</table>\n\n\nWe declare under the penalties of perjury that this certificate has been made in good faith, verified by us, and to the best of our knowledge and belief, is true and\ncorrect, pursuant to the provisions of the National Internal Revenue Code, as amended, and the regulations issued under authority thereof. Further, we give our consent\nto the processing of our information as contemplated under the *Data Privacy Act of 2012 (R.A. No. 10173) for legitimate and lawful purposes.\n\nPABLITO A. PAMANTANG, JR. / FINANCE MANAGER / 198-656-147-000\n\nSignature over Panted Name of Payor/Payor\'s Authorized Representative/Tax Agent\n(Indicate Title/Designation and TIN)\n\nTax Agent Accreditation No./\nAttorney\'s Roll No. (if applicable)\n\nDate of Iss\n(MM/DD/Y\n\nDate of Expir\n\n(MM/DD/YY)\n\nCONFORME:\n\nSignature over Printed Name of Payee/Payee\'s Authorized Representative/Tax Agent\n\n(Indicate Title/Designation and TIN)\n\nTax Agent Accreditation No./\nAttorney\'s Roll No. (if applicable)\n\nDate of Iss\n(MM/DD/Y\n\nDate of Expir\n(MM/DD/YY)\n\n*NOTE: The BIR Data Privacy is in the BIR website (www.bir.gov.ph)\n'

In [10]:
system_prompt = f"""You are an AI assistant that extracts data from documents."""

In [11]:
# Prepare the user content for the OpenAI API including any specific details for processing this type of document, text, and the document page images.
user_content = []

In [12]:
user_text_prompt = """Extract the data from this document. 
- If a value is not present, provide null.
- Some values must be inferred based on the rules defined in the policy.
- Dates should be in the format YYYY-MM-DD."""

user_content.append({
    "type": "text",
    "text": user_text_prompt
})

user_content.append({
    "type": "text",
    "text": markdown
})

In [13]:
import io
import base64

def encode_page(page):
    byte_io = io.BytesIO()
    page.save(byte_io, format='PNG')
    base64_data = base64.b64encode(byte_io.getvalue()).decode('utf-8')
    return {
        "type": "image_url",
        "image_url": {
            "url": f"data:image/png;base64,{base64_data}"
        }
    }

with Stopwatch() as image_stopwatch:
    with open(pdf_fpath, "rb") as f:
        document_bytes = f.read()

    pages = convert_from_bytes(document_bytes)
    
    # Process each page in parallel using multiple processes
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(encode_page, pages))
        user_content.extend(results)

In [14]:
with Stopwatch() as oai_stopwatch:
    completion = openai_client.beta.chat.completions.parse(
        model=settings.gpt4o_model_deployment_name,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        response_format=Bir2307,
        max_tokens=4096,
        temperature=0.1,
        top_p=0.1,
        logprobs=True # Enabled to determine the confidence of the response.
    )

In [16]:
completion2 = completion

### Understanding the Structured Outputs JSON schema

Using [Pydantic's JSON schema feature](https://docs.pydantic.dev/latest/concepts/json_schema/), the [BIR2307](../../modules/samples/models/bir2307.py) data model is automatically converted to a JSON schema when applied to the `response_format` parameter of the OpenAI chat completions request.

The JSON schema is used to instruct the GPT-4o model to generate a strict output that adheres to the structure defined. The approach using Pydantic makes it easier for developers to manage the data structure in code, with helpful descriptions and examples that will be included in the final JSON schema.

Demonstrated below, you can see how the BIR2307 data model is understood by the OpenAI request:


In [17]:
print(json.dumps(Bir2307.model_json_schema(), indent=2))

{
  "$defs": {
    "BIRForm": {
      "properties": {
        "formNumber": {
          "$ref": "#/$defs/DataField",
          "description": "BIR form number"
        },
        "version": {
          "$ref": "#/$defs/DataField",
          "description": "BIR form version"
        }
      },
      "required": [
        "formNumber",
        "version"
      ],
      "title": "BIRForm",
      "type": "object"
    },
    "Certificate": {
      "properties": {
        "title": {
          "$ref": "#/$defs/DataField",
          "description": "Certificate title"
        },
        "code": {
          "$ref": "#/$defs/DataField",
          "description": "Certificate code"
        }
      },
      "required": [
        "title",
        "code"
      ],
      "title": "Certificate",
      "type": "object"
    },
    "DataField": {
      "description": "A generic field to hold a value and its type.",
      "properties": {
        "value": {
          "anyOf": [
            {
              "typ

## Visualize the outputs

To provide context for the execution of the code, the following code blocks visualize the outputs of the data extraction process.

This includes:

- The accuracy of the structured data extraction comparing the expected output with the output generated by Azure OpenAI's GPT-4o model.
- The confidence score of the structured data extraction based on combining the confidence scores of the Azure AI Document Intelligence layout analysis and the log probability of the output generated by Azure OpenAI's GPT-4o model.
- The execution time of the end-to-end process.
- The total number of tokens consumed by the GPT-4o model.
- The side-by-side comparison of the expected output and the output generated by Azure OpenAI's GPT-4o model.

### Understanding Accuracy vs Confidence

When using AI to extract structured data, both confidence and accuracy are essential for different but complementary reasons.

- **Accuracy** measures how close the AI model's output is to a ground truth or expected output. It reflects how well the model's predictions align with reality.
  - Accuracy ensures consistency in the extraction process, which is crucial for downstream tasks using the data.
- **Confidence** represents the AI model's internal assessment of how certain it is about its predictions.
  - Confidence indicates that the model is certain about its predictions, which can be a useful indicator for human reviewers to step in for manual verification.

High accuracy and high confidence are ideal, but in practice, there is often a trade-off between the two. While accuracy cannot always be self-assessed, confidence scores can and should be used to prioritize manual verification of low-confidence predictions.

In [18]:
# Displays the output of the Azure AI Document Intelligence pre-built layout analysis in Markdown format.
display(Markdown(markdown))

<!-- PageHeader="TS-WF-205F-0012327" -->


<figure>

Republic of the Philippines
Department of Finance
Bureau of Internal Revenue

</figure>


For BIR BCS/
Use Only Item:

BIR Form No.
2307
January 2018 (ENCS)


# Certificate of Creditable Tax Withheld At Source

2307 01/18ENCS

Fill in all applicable spaces. Mark all appropriate boxes with an "X".

1 For the Period
From
07012023

(MM/DD/YY)

To
09302023

(MM/DD/YY)


## Part I - Payee Information

2 Taxpayer Identification Number (TIN)

267-090-070-00000

3 Payee's Name (Last Name, First Name, Middle Name for Individual OR Registered Name for Non-Individuals)

THERMA MARINE, INC.

4 Registered Address
MOBILE 2, LAWIS, SANTA ANA, AGUSAN DEL NORTE PHILIPPINES 8602 PHILIPPINES

4A ZIP Code

8602

5 Foreign Address, if applicable


## Part II - Payor Information

6 Taxpayer Identification Number (TIN)

008-657-558-0000

7 Payor's Name (Last Name, First Name, Middle Name for Individual OR Registered Name for Non-Individuals)
ANGAT HYDROPOWER CORPORATION

8 Registered Address
ANGAT HYDROELECTRIC POWER PLANT SAN LORENZO, NORZAGARAY, BULACAN

8A ZIP Code

3013


<table>
<tr>
<th colspan="7">Part III - Details of Monthly Income Payments and Taxes Withheld</th>
</tr>
<tr>
<th rowspan="2">Income Payments Subject to Expanded Withholding Tax</th>
<th rowspan="2">ATC</th>
<th colspan="4">AMOUNT OF INCOME PAYMENTS</th>
<th rowspan="2">Tax Withheld for the Quarter</th>
</tr>
<tr>
<th>1st Month of the Quarter</th>
<th>2nd Month of the Quarter</th>
<th>3rd Month of the Quarter</th>
<th>Total</th>
</tr>
<tr>
<td>EWT- Income payments made by top 10,000 private corporations to their local/resident supplier of services</td>
<td>WC 160</td>
<td></td>
<td>1.22</td>
<td></td>
<td>1.22</td>
<td>0.02</td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td>Total</td>
<td></td>
<td></td>
<td>1.22</td>
<td></td>
<td>1.22</td>
<td>0.02</td>
</tr>
<tr>
<td>Money Payments Subject to Withholding of Business Tax (Government &amp; Private)</td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td>Total</td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


We declare under the penalties of perjury that this certificate has been made in good faith, verified by us, and to the best of our knowledge and belief, is true and
correct, pursuant to the provisions of the National Internal Revenue Code, as amended, and the regulations issued under authority thereof. Further, we give our consent
to the processing of our information as contemplated under the *Data Privacy Act of 2012 (R.A. No. 10173) for legitimate and lawful purposes.

PABLITO A. PAMANTANG, JR. / FINANCE MANAGER / 198-656-147-000

Signature over Panted Name of Payor/Payor's Authorized Representative/Tax Agent
(Indicate Title/Designation and TIN)

Tax Agent Accreditation No./
Attorney's Roll No. (if applicable)

Date of Iss
(MM/DD/Y

Date of Expir

(MM/DD/YY)

CONFORME:

Signature over Printed Name of Payee/Payee's Authorized Representative/Tax Agent

(Indicate Title/Designation and TIN)

Tax Agent Accreditation No./
Attorney's Roll No. (if applicable)

Date of Iss
(MM/DD/Y

Date of Expir
(MM/DD/YY)

*NOTE: The BIR Data Privacy is in the BIR website (www.bir.gov.ph)


In [19]:
bir_2307 = completion.choices[0].message.parsed

In [20]:
bir_2307.model_dump()

# Make the dump json beautify
print(json.dumps(bir_2307.model_dump(), indent=2))

{
  "pageHeader": {
    "value": "TS-WF-205F-0012327",
    "type": "string"
  },
  "governmentInformation": {
    "country": {
      "value": "Philippines",
      "type": "string"
    },
    "department": {
      "value": "Department of Finance",
      "type": "string"
    },
    "agency": {
      "value": "Bureau of Internal Revenue",
      "type": "string"
    }
  },
  "birForm": {
    "formNumber": {
      "value": "2307",
      "type": "string"
    },
    "version": {
      "value": "January 2018 (ENCS)",
      "type": "string"
    }
  },
  "certificate": {
    "title": {
      "value": "Certificate of Creditable Tax Withheld At Source",
      "type": "string"
    },
    "code": {
      "value": "2307 01/18ENCS",
      "type": "string"
    }
  },
  "period": {
    "from_": {
      "value": "2023-07-01",
      "type": "string"
    },
    "to": {
      "value": "2023-09-30",
      "type": "string"
    },
    "format": {
      "value": "MM/DD/YY",
      "type": "string"
    }
  },
  "

In [ ]:
bir_2307_dict = bir_2307.model_dump()
expected_dict = expected.model_dump()

In [25]:
# Determines the accuracy of the extracted data against the expected values.
accuracy = bir_2307_evaluator.evaluate(expected=expected_dict, actual=bir_2307_dict)

In [26]:
# Determines the confidence of the extracted data using both the OpenAI and Azure Document Intelligence responses.
di_confidence = evaluate_di_confidence(bir_2307_dict, result)
oai_confidence = evaluate_openai_confidence(bir_2307_dict, completion.choices[0])

confidence = merge_confidence_values(di_confidence, oai_confidence)

In [27]:
# Gets the total execution time of the data extraction process.
total_elapsed = di_stopwatch.elapsed + image_stopwatch.elapsed + oai_stopwatch.elapsed

# Gets the prompt tokens and completion tokens from the completion response.
prompt_tokens = completion.usage.prompt_tokens
completion_tokens = completion.usage.completion_tokens

In [31]:
# Save the output of the data extraction result.
extraction_result = DataExtractionResult(bir_2307_dict, confidence, accuracy, prompt_tokens, completion_tokens, total_elapsed)


sample_path = f"{working_dir}/results/"
sample_name = "bir_2307"
create_json_file(f"{sample_path}/{sample_name}.{pdf_fname}.json", extraction_result)

In [30]:
# Display the outputs of the data extraction process.
import pandas as pd

df = pd.DataFrame([
    {
        "Accuracy": f"{accuracy['overall'] * 100:.2f}%",
        "Confidence": f"{confidence['_overall'] * 100:.2f}%",
        "Execution Time": f"{total_elapsed:.2f} seconds",
        "Document Intelligence Execution Time": f"{di_stopwatch.elapsed:.2f} seconds",
        "Image Pre-processing Execution Time": f"{image_stopwatch.elapsed:.2f} seconds",
        "OpenAI Execution Time": f"{oai_stopwatch.elapsed:.2f} seconds",
        "Prompt Tokens": prompt_tokens,
        "Completion Tokens": completion_tokens
    }
])

display(df)
display(get_extraction_comparison(expected_dict, bir_2307_dict, confidence, accuracy['accuracy']))

,Accuracy,Confidence,Execution Time,Document Intelligence Execution Time,Image Pre-processing Execution Time,OpenAI Execution Time,Prompt Tokens,Completion Tokens
0,100.00%,97.86%,62.76 seconds,8.47 seconds,0.24 seconds,54.05 seconds,6670,708


,Field,Expected,Extracted,Confidence,Accuracy
0,birForm_formNumber_type,string,string,95.77%,Match
1,birForm_formNumber_value,2307,2307,98.40%,Match
2,birForm_version_type,string,string,100.00%,Match
3,birForm_version_value,January 2018 (ENCS),January 2018 (ENCS),98.95%,Match
4,certificate_code_type,string,string,100.00%,Match
5,certificate_code_value,2307 01/18ENCS,2307 01/18ENCS,98.90%,Match
6,certificate_title_type,string,string,100.00%,Match
7,certificate_title_value,Certificate of Creditable Tax Withheld At Source,Certificate of Creditable Tax Withheld At Source,99.75%,Match
8,dateOfExpiry_type,null,null,100.00%,Match
9,dateOfExpiry_value,None,None,0.00%,Match
